The purpose of this notebook is to evaluate the ML solution to estimating joint PDFs of aerosol size and concentration

Baseline NLL against (coarse) histograms?

use the training data to estimate histgrams with bin widths optimized using validation data.
Then evaluate the NLL of those PDFs on the test data.  Compare the result to the ML solution.
Building the histograms requires we do a grid search over all possible inputs

In [3]:
import os
import sys
import torch
# import logging
import torch.nn as nn
# import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
# from torch.autograd import Variable
# import torchvision.models as models

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from scipy.special import gamma, gammaln

import copy
import datetime
import yaml

import importlib

# import tqdm

In [4]:
dirP_str = os.path.join(os.environ['HOME'], 
                    'PythonScripts',
                    'aerosol-scattering',
                    'library')
if dirP_str not in sys.path:
    sys.path.append(dirP_str)
    
import evid_nn
import data

In [5]:
is_cuda = torch.cuda.is_available()
device = torch.device(torch.cuda.current_device()) if is_cuda else torch.device("cpu")

if is_cuda:
    torch.backends.cudnn.benchmark = True

print(f'Preparing to use device {device}')

Preparing to use device cuda:0


In [6]:
model_path = "/export/breeze1/mhayman/aerosol_poly_nn/models/"

In [7]:
# train model on refractive index from 1.3-1.5 and 0.0-0.06
model_str_lst = [
    # "20260310T065310",  # 6 beta, 2 alpha all refractive indices
    "20260311T110300",  # 3 beta, 2 alpha all refractive indices
    # "20260324T070054",  # 1 beta, 1 alpha 355 nm, all refractive indices
    # "20260327T080000",  # 1 beta, 1 alpha 532 nm, all refractive indices
]

In [8]:
example_count = 10

# idx_arr = np.array([18334])
# idx_arr = np.array([60308, 11462, 44852,  6123, 41998, 50674, 87012, 87725, 87707, 17718])
idx_arr = np.array([60308, 11462, 44852,  6123, 41998, 50674, 87012, 87725, 87707, 18334, 44305, 44535, 49700, 55717, 55721, 55744,
        55875, 55960, 56042, 56201, 56414, 56527, 56740, 56798, 56810,
        56822, 56823, 62837, 65213, 53289])

# Create Histogram PDFs

In [11]:
# hist_pt_range = np.logspace(1,3,100)
hist_pt_range = np.array([10])


label_param_lst_lst = []
label_wl_lst_lst = []

for m_idx, model_str in enumerate(model_str_lst):
    print(f"loading model {m_idx+1} of {len(model_str_lst)}")
    conf_reload, model_reload = evid_nn.load_poly_nn_model(model_str,name_str='hyper_best',device=device,path=model_path)

    x_scaler = evid_nn.rescale(0,1)
    x_scaler.set_config(**conf_reload['save_model']['x_scaler'])
    y_scaler = evid_nn.rescale(0,1)
    y_scaler.set_config(**conf_reload['save_model']['y_scaler'])

    # x_rescale_lst.append(x_scaler)
    # y_rescale_lst.append(y_scaler)

    # conf_lst.append(conf_reload)
    conf = conf_reload
    dtype = model_reload.dtype

    nc_attrs = {'model':model_str}

    # if m_idx == 0:
    #     intrp_tnsr = torch.tensor(percentile_lst,dtype=dtype,device=device)

    
    label_param_lst = []
    label_wl_lst = []
    
    if 'extinction_coefficient' in conf_reload['data']['wavelength_inputs']:
        label_param_lst.append(r"$\alpha$")
        wl_key='extinction_coefficient'
    if 'backscatter_coefficient' in conf_reload['data']['wavelength_inputs']:
        label_param_lst.append(r"$\beta$")
        wl_key='backscatter_coefficient'

    for wl in conf_reload['data']['wavelength_inputs'][wl_key]['wavelength_lst']:
        label_wl_lst.append(str(int(wl*1e9)))

    label_param_lst_lst.append(label_param_lst)
    label_wl_lst_lst.append(label_wl_lst)


    ### Load Data ###
    # this is repeated because rescaling could theoretically be different between models
    
    override_validation = False  # forces the validation data to be split from training

    rank = 0
    trial = False
    
    
    
    ds = xr.open_dataset(os.path.join(conf['data']['data_path'],conf['data']['data_file']))
    
    train_batch_size = conf['data']['batch_size']
    valid_batch_size = conf['data']['batch_size']
    test_batch_size = conf['data']['batch_size']
    
    train_idx = int(np.round(ds.sizes['data_index']*conf['data']['train_fraction']))
    valid_idx = train_idx+int(np.round(ds.sizes['data_index']*conf['data']['valid_fraction']))
    test_idx = train_idx+int(np.round(ds.sizes['data_index']*conf['data']['test_fraction']))
    
    
    train_input_arr, train_label_arr, input_str_lst, output_str_lst, input_frac_uncertainty = data.build_training_array(ds,conf)
    
    if not override_validation:
        valid_ds = xr.open_dataset(os.path.join(conf['data']['valid_data_path'],conf['data']['valid_data_file']))
        
        
        # valid_label_arr = xr.concat(output_lst,pd.Index(conf['data']['output_cols'],name='label_vars'))
        valid_input_arr, valid_label_arr, _, _, _ = data.build_training_array(valid_ds,conf)
        
        x_train = x_scaler.scale(np.log(train_input_arr.values))
        x_valid = x_scaler.scale(np.log(valid_input_arr.values))
        
        y_train = y_scaler.scale(np.log(train_label_arr.values))
        y_valid = y_scaler.scale(np.log(valid_label_arr.values))
    else:
        print("validation dataset encountered an error:")
        print("override_validation=True, so validation is split from training data")
        # print(str(E))
        x_train = x_scaler.scale(np.log(train_input_arr.values[:train_idx,:]))
        x_valid = x_scaler.scale(np.log(train_input_arr.values[train_idx:valid_idx,:]))
        
        y_train = y_scaler.scale(np.log(train_label_arr.values[:train_idx,:]))
        y_valid = y_scaler.scale(np.log(train_label_arr.values[train_idx:valid_idx,:]))


    for hist_in_pt in hist_pt_range:
    
        # 1. Digitize Inputs: Multiply by bin count and floor to get an integer index.
        # Using np.clip prevents exactly 1.0 from spilling into an out-of-bounds bin index.
        x_bins = np.clip(np.floor(x_train * hist_in_pt).astype(int), 0, hist_in_pt - 1)
        
        for hist_out_pt in hist_pt_range:
            
            # 2. Digitize Outputs
            y_bins = np.clip(np.floor(y_train * hist_out_pt).astype(int), 0, hist_out_pt - 1)
            
            # 3. Stack inputs and outputs into a single array of shape (N_samples, D_in + D_out)
            combined_bins = np.hstack((x_bins, y_bins))
            
            # 4. Find all unique populated bins and count their frequencies
            # This completely skips any empty bins in your high-dimensional grid.
            unique_combinations, counts = np.unique(combined_bins, axis=0, return_counts=True)
            
            # unique_combinations[:, :D_in] represents the input bin coordinates
            # unique_combinations[:, D_in:] represents the output bin coordinates
            # counts contains the number of data points inside each specific joint bin
            
            # --- VALIDATION/SCORING LOGIC GOES HERE ---
            # e.g., mapping these counts to probabilities to evaluate against the validation set

    # # evaluate a combination of input and output resolutions to find the optimal combo
    # for hist_in_pt in hist_pt_range:
    #     h_in_bins = np.linspace(0,1,hist_in_pt)
        
    #     for hist_out_pt in hist_pt_range:
    #         h_out_bins = np.linspace(0,1,hist_out_pt)

    #         # now we need to loop over all combinations of inputs determined by h_in_bins for each input and store the histogram
    #         # resulting from it.
            

loading model 1 of 1
loading model from 
/export/breeze1/mhayman/aerosol_poly_nn/models/


NameError: name 'percentile_lst' is not defined

In [ ]:
conf_lst = []

r_centroid_arr_lst = []
n_centroid_arr_lst = []
r2_centroid_arr_lst = []
n2_centroid_arr_lst = []
r_act_arr_lst = []
n_act_arr_lst = []
p_model_arr_lst = []

x_rescale_lst = []
y_rescale_lst = []

label_param_lst_lst = []
label_wl_lst_lst = []

r_interval_arr_lst = []
n_interval_arr_lst = []

r_sim_hist_lst_lst = []
n_sim_hist_lst_lst = []

sim_hist_in_data_lst_lst = []

f_pdf_arr_lst = []
f_pdf_ens_arr_lst = []

r_ens_centroid_arr_lst = []
n_ens_centroid_arr_lst = []
r2_ens_centroid_arr_lst = []
n2_ens_centroid_arr_lst = []
r_ens_interval_arr_lst = []
n_ens_interval_arr_lst = []


for m_idx, model_str in enumerate(model_str_lst):
    print(f"loading model {m_idx+1} of {len(model_str_lst)}")
    conf_reload, model_reload = evid_nn.load_poly_nn_model(model_str,name_str='hyper_best',device=device,path=model_path)

    x_scaler = evid_nn.rescale(0,1)
    x_scaler.set_config(**conf_reload['save_model']['x_scaler'])
    y_scaler = evid_nn.rescale(0,1)
    y_scaler.set_config(**conf_reload['save_model']['y_scaler'])

    x_rescale_lst.append(x_scaler)
    y_rescale_lst.append(y_scaler)

    conf_lst.append(conf_reload)
    conf = conf_reload
    dtype = model_reload.dtype

    nc_attrs = {'model':model_str}

    if m_idx == 0:
        intrp_tnsr = torch.tensor(percentile_lst,dtype=dtype,device=device)

    
    label_param_lst = []
    label_wl_lst = []
    
    if 'extinction_coefficient' in conf_reload['data']['wavelength_inputs']:
        label_param_lst.append(r"$\alpha$")
        wl_key='extinction_coefficient'
    if 'backscatter_coefficient' in conf_reload['data']['wavelength_inputs']:
        label_param_lst.append(r"$\beta$")
        wl_key='backscatter_coefficient'

    for wl in conf_reload['data']['wavelength_inputs'][wl_key]['wavelength_lst']:
        label_wl_lst.append(str(int(wl*1e9)))

    label_param_lst_lst.append(label_param_lst)
    label_wl_lst_lst.append(label_wl_lst)


    ### Load Data ###
    # this is repeated because rescaling could theoretically be different between models
    
    override_validation = False  # forces the validation data to be split from training

    rank = 0
    trial = False
    
    
    
    ds = xr.open_dataset(os.path.join(conf['data']['data_path'],conf['data']['data_file']))
    
    train_batch_size = conf['data']['batch_size']
    valid_batch_size = conf['data']['batch_size']
    test_batch_size = conf['data']['batch_size']
    
    train_idx = int(np.round(ds.sizes['data_index']*conf['data']['train_fraction']))
    valid_idx = train_idx+int(np.round(ds.sizes['data_index']*conf['data']['valid_fraction']))
    test_idx = train_idx+int(np.round(ds.sizes['data_index']*conf['data']['test_fraction']))
    
    
    train_input_arr, train_label_arr, input_str_lst, output_str_lst, input_frac_uncertainty = data.build_training_array(ds,conf)
    
    if not override_validation:
        valid_ds = xr.open_dataset(os.path.join(conf['data']['valid_data_path'],conf['data']['valid_data_file']))
        
        
        # valid_label_arr = xr.concat(output_lst,pd.Index(conf['data']['output_cols'],name='label_vars'))
        valid_input_arr, valid_label_arr, _, _, _ = data.build_training_array(valid_ds,conf)
        
        x_train = x_scaler.scale(np.log(train_input_arr.values))
        x_valid = x_scaler.scale(np.log(valid_input_arr.values))
        
        y_train = y_scaler.scale(np.log(train_label_arr.values))
        y_valid = y_scaler.scale(np.log(valid_label_arr.values))
    else:
        print("validation dataset encountered an error:")
        print("override_validation=True, so validation is split from training data")
        # print(str(E))
        x_train = x_scaler.scale(np.log(train_input_arr.values[:train_idx,:]))
        x_valid = x_scaler.scale(np.log(train_input_arr.values[train_idx:valid_idx,:]))
        
        y_train = y_scaler.scale(np.log(train_label_arr.values[:train_idx,:]))
        y_valid = y_scaler.scale(np.log(train_label_arr.values[train_idx:valid_idx,:]))
    
    
    cond_args = conf['data']['cond_args']
    
    train_dataset = evid_nn.EvidDataset(x_train,y_train,
                                dtype=dtype,device=device,
                                **cond_args)
    valid_dataset = evid_nn.EvidDataset(x_valid,y_valid,
                                dtype=dtype,device=device,
                                **cond_args)
    # test_dataset = evid_nn.EvidDataset(x_test,y_test,
    #                             dtype=dtype,device=device,
    #                             **cond_args)
    
    train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
    valid_dataloader = DataLoader(valid_dataset, batch_size=valid_batch_size, shuffle=True)

    # use the training data to estimate histgrams with bin widths optimized using validation data
    # then evaluate the NLL of those PDFs on the test data.  Compare the result to the ML solution.
    # building the histograms requires we do a grid search over all possible inputs
    
    
    test_ds = xr.open_dataset(os.path.join(conf['data']['test_data_path'],conf['data']['test_data_file']))
    
    test_input_arr, test_label_arr, _, _, _ = data.build_training_array(test_ds,conf)
    
    x_test = x_scaler.scale(np.log(test_input_arr.values))
    
    y_test = y_scaler.scale(np.log(test_label_arr.values))
    
    test_dataset = evid_nn.EvidDataset(x_test,y_test,
                                dtype=dtype,device=device,
                                **cond_args)
    
    test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

    ### End Data Loader ###

    # in_data = x_train
    # out_label = y_train
    
    # in_data = x_valid
    # out_label = y_valid

    ### Perform example analysis ###
    # actual sample data
    in_data = x_test
    out_label = y_test

    # data for histogram analysis
    hist_in_data = np.concatenate([x_train,x_valid,x_test],axis=0)
    hist_out_label = np.concatenate([y_train,y_valid,y_test],axis=0)

    if m_idx == 0:
        # idx_arr = (np.random.rand(example_count)*in_data.shape[0]).astype(int)  # get random input
        r_act = out_label[:,1][idx_arr]
        n_act = out_label[:,0][idx_arr]
        
    sim_idx_lst = []
    r_sim_hist_lst = []
    n_sim_hist_lst = []
    sim_hist_in_data_lst = []
    for sidx in range(idx_arr.size):
        idx = idx_arr[sidx]
        sim_idx = np.where(np.prod(np.abs(hist_in_data - in_data[idx,:].reshape(1,-1)) < hist_width,axis=1))
        sim_idx_lst.append(sim_idx)

        sim_hist_in_data_lst.append(hist_in_data[sim_idx])

        r_sim_hist = hist_out_label[:,1][sim_idx]
        n_sim_hist = hist_out_label[:,0][sim_idx]

        r_sim_hist_lst.append(r_sim_hist)
        n_sim_hist_lst.append(n_sim_hist)

    r_sim_hist_lst_lst.append(r_sim_hist_lst)
    n_sim_hist_lst_lst.append(n_sim_hist_lst)
    sim_hist_in_data_lst_lst.append(sim_hist_in_data_lst)

    # pdf_ax = np.exp(y_scaler.unscale(np.stack([x_plt,x_plt],axis=1)))
    # hist_ax = np.exp(y_scaler.unscale(np.stack([h_part[1],h_part[2]],axis=1)))
    # act_ax = np.exp(y_scaler.unscale(np.stack([np.array([n_act]),np.array([r_act])],axis=1))).squeeze()

    # pdf_ax_lst.append(pdf_ax)
    # hist_ax_lst.append(hist_ax)
    # act_ax_lst.append(act_ax)
        

    f_pdf_lst = []
    f_pdf_ens_lst = []
    with torch.no_grad():
        for sidx, sim_idx in enumerate(sim_idx_lst):
            idx = idx_arr[sidx]
            f_pdf_tnsr = model_reload.output_pdf(torch.tensor(in_data[idx:(idx+1),:],dtype=dtype,device=device))
            f_pdf = f_pdf_tnsr.detach().cpu().numpy()
            x_plt = model_reload.x_int.detach().cpu().numpy()
            
            ensemble_size = 100
            ensemble_width = hist_width # (observation uncertainty)
            loop_length = 4
            f_pdf_ens = np.zeros((f_pdf.shape[1],f_pdf.shape[2]))
            for ens_group_idx in range(loop_length):
                f_pdf_ens_tnsr = model_reload.output_pdf(torch.tensor(in_data[idx:(idx+1),:]+(np.random.rand(ensemble_size,in_data.shape[1])*2-1)*ensemble_width,dtype=dtype,device=device))
                f_pdf_ens += np.sum(f_pdf_ens_tnsr.detach().cpu().numpy(),axis=0)
                # print(ens_group_idx)
            
            f_pdf_ens = f_pdf_ens/(loop_length*ensemble_size)

            f_pdf_lst.append(f_pdf)
            f_pdf_ens_lst.append(f_pdf_ens)

    f_pdf_arr_lst.append(np.concatenate(f_pdf_lst,axis=0))
    f_pdf_ens_arr_lst.append(np.stack(f_pdf_ens_lst,axis=0))

In [ ]:
custom_label_lst = [r'2$\alpha$ 6$\beta$',r'2$\alpha$ 3$\beta$', r'355 nm',r'532 nm']

In [ ]:
ex_idx = 15
n_col = 4
n_row = int(np.ceil(len(f_pdf_arr_lst)/n_col))
fig,ax_lst = plt.subplots(n_row,n_col,figsize=(5*n_col,5*n_row))
ax_lst = ax_lst.ravel()
for m_idx, hn_2d in enumerate(f_pdf_arr_lst):
    pdf_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([x_plt,x_plt],axis=1)))
    # f_pdf = f_pdf_ens_arr_lst[m_idx][ex_idx,...]
    f_pdf = f_pdf_arr_lst[m_idx][ex_idx,...]
    ax = ax_lst[m_idx]
    ax.pcolormesh(pdf_ax[:,1],pdf_ax[:,0],f_pdf)
    # ax.pcolormesh(pdf_ax[:,1],pdf_ax[:,0],f_pdf[0,...]/np.sum(f_pdf[0,...]))
    ax.set_xscale('log')
    ax.set_yscale('log')
    if m_idx//n_col == n_row-1:
        ax.set_xlabel(r'effective radius $m$')
    if m_idx%n_col == 0:
        ax.set_ylabel(r'number concentration $cm^{-3}$')
    title = custom_label_lst[m_idx]
    # title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])
    ax.set_title(title)

In [ ]:
n_col = 4
n_row = int(np.ceil(len(f_pdf_ens_arr_lst)/n_col))
fig,ax_lst = plt.subplots(n_row,n_col,figsize=(5*n_col,5*n_row))
ax_lst = ax_lst.ravel()
for m_idx, hn_2d in enumerate(f_pdf_ens_arr_lst):
    pdf_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([x_plt,x_plt],axis=1)))
    f_pdf = f_pdf_ens_arr_lst[m_idx][ex_idx,...]
    ax = ax_lst[m_idx]
    ax.pcolormesh(pdf_ax[:,1],pdf_ax[:,0],f_pdf)
    # ax.pcolormesh(pdf_ax[:,1],pdf_ax[:,0],f_pdf[0,...]/np.sum(f_pdf[0,...]))
    ax.set_xscale('log')
    ax.set_yscale('log')
    if m_idx//n_col == n_row-1:
        ax.set_xlabel(r'effective radius $m$')
    if m_idx%n_col == 0:
        ax.set_ylabel(r'number concentration $cm^{-3}$')
    title = custom_label_lst[m_idx]
    # title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])
    ax.set_title(title)

In [ ]:
n_col = 4
n_row = int(np.ceil(len(y_rescale_lst)/n_col))
fig,ax_lst = plt.subplots(n_row,n_col,figsize=(5*n_col,5*n_row))
ax_lst = ax_lst.ravel()
for m_idx, y_scaler in enumerate(y_rescale_lst):
    h_part = np.histogram2d(r_sim_hist_lst_lst[m_idx][ex_idx],n_sim_hist_lst_lst[m_idx][ex_idx],bins=np.linspace(0,1,500))
    hist_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([h_part[1],h_part[2]],axis=1)))
    # pdf_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([x_plt,x_plt],axis=1)))
    # f_pdf = f_pdf_arr_lst[m_idx][ex_idx,...]
    ax = ax_lst[m_idx]
    ax.pcolormesh(hist_ax[:,1],hist_ax[:,0],h_part[0].T/np.sum(h_part[0]))

    # add ensemble contour
    pdf_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([x_plt,x_plt],axis=1)))
    f_pdf = f_pdf_ens_arr_lst[m_idx][ex_idx,...]
    contours = ax.contour(pdf_ax[:,1],pdf_ax[:,0], f_pdf, levels = 3, colors='white', linestyles=':', linewidths=1)
    
    # ax.pcolormesh(pdf_ax[:,1],pdf_ax[:,0],f_pdf[0,...]/np.sum(f_pdf[0,...]))
    ax.set_xscale('log')
    ax.set_yscale('log')
    if m_idx//n_col == n_row-1:
        ax.set_xlabel(r'effective radius $m$')
    if m_idx%n_col == 0:
        ax.set_ylabel(r'number concentration $cm^{-3}$')
    title = custom_label_lst[m_idx]
    # title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])
    ax.set_title(title)

In [ ]:
m_idx = 0  # model index
# ex_idx = 8 # -1  # example index
hist_out_label

# hist_ax = np.exp(y_rescale_lst[-1].unscale(np.stack([0.5*(h_part_total[1][:-1]+h_part_total[1][1:]),
#                                                 0.5*(h_part_total[2][:-1]+h_part_total[2][1:])],axis=1)))

hist_ax = np.exp(y_rescale_lst[-1].unscale(np.stack([0.5*(h_part_total[1][:-1]+h_part_total[1][1:]),
                                                0.5*(h_part_total[2][:-1]+h_part_total[2][1:])],axis=1)))

fig,ax = plt.subplots(1,1,figsize=(5,5))
# ax = ax_lst[m_idx]
# f_pdf = f_pdf_arr_lst[m_idx][ex_idx,...] # direct output
f_pdf = f_pdf_ens_arr_lst[m_idx][ex_idx,...]  # ensemble output
ax.pcolormesh(hist_ax[:,1],hist_ax[:,0],h_part_total[0].T/np.sum(h_part_total[0]))
contours = ax.contour(pdf_ax[:,1],pdf_ax[:,0], f_pdf, levels = 3, colors='white', linestyles=':', linewidths=1)
ax.set_xscale('log')
ax.set_yscale('log')
# if m_idx//n_col == 1:
ax.set_xlabel(r'mean radius $m$')
# if m_idx%n_col == 0:
ax.set_ylabel(r'number concentration $cm^{-3}$')
# title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])

title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])
ax.set_title("Prior and "+title)

In [ ]:
ex_idx = 20
n_col = 4
n_row = int(np.ceil(len(y_rescale_lst)/n_col))
fig,ax_lst = plt.subplots(n_row,n_col,figsize=(5*n_col,5*n_row))
ax_lst = ax_lst.ravel()
for m_idx, hn_2d in enumerate(y_rescale_lst):
    pdf_ax = np.exp(y_rescale_lst[m_idx].unscale(np.stack([x_plt,x_plt],axis=1)))
    # f_pdf = f_pdf_arr_lst[m_idx][ex_idx,...]
    f_pdf = f_pdf_ens_arr_lst[m_idx][ex_idx,...]
    ax = ax_lst[m_idx]
    ax.pcolormesh(hist_ax[:,1],hist_ax[:,0],h_part_total[0].T/np.sum(h_part_total[0]))
    contours = ax.contour(pdf_ax[:,1],pdf_ax[:,0], f_pdf, levels = 3, colors='white', linestyles=':', linewidths=1)
    ax.set_xscale('log')
    ax.set_yscale('log')
    if m_idx//n_col == 1:
        ax.set_xlabel(r'mean radius $m$')
    if m_idx%n_col == 0:
        ax.set_ylabel(r'number concentration $cm^{-3}$')
    title = ', '.join(label_param_lst_lst[m_idx])+": "+", ".join(label_wl_lst_lst[m_idx])
    ax.set_title(title)